# **AirBnB Open Data**

Airbnb, Inc is an American company that operates an online marketplace for lodging, primarily homestays for vacation rentals, and tourism activities. Based in San Francisco, California, the platform is accessible via website and mobile app. Airbnb does not own any of the listed properties; instead, it profits by receiving commission from each booking. The company was founded in 2008. Airbnb is a shortened version of its original name, AirBedandBreakfast.com.


## **Context**

Since 2008, guests and hosts have used Airbnb to travel in a more unique, personalized way. As part of the Airbnb Inside initiative, this dataset describes the listing activity of homestays in New York City

In [ ]:
import pandas as pd
import numpy as np
from data_prep_functions import remove_dollar_sign
import duckdb

df = pd.read_csv('Airbnb_Open_Data.csv')
df.columns = [col.lower().replace(' ', '_').replace('.', '') for col in df.columns]

C:\Users\OSVALDO-SOFTENG\AppData\Local\Temp\ipykernel_20540\881112542.py:6: DtypeWarning: Columns (0: license) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Airbnb_Open_Data.csv')


### Cleaning Columns

In [2]:
print(df.columns)

# Cleaning Unnamed:0 (index) and license
df = df.drop(columns=['license'])
df.describe()

# The columns minimum_nights and availability_365 has negative numbers, which are not possible in a real case. We should remove it right away by taking the absolute value of the negative numbers.
df['availability_365'] = np.where(df['availability_365'] < 0, abs(df['availability_365']), df['availability_365'])
df['minimum_nights'] = np.where(df['minimum_nights'] < 0, abs(df['minimum_nights']), df['minimum_nights'])

# The price and service fee still has dollar sign, so we should remove it.
df['price'] = df['price'].apply(lambda x: remove_dollar_sign(x))
df['service_fee'] = df['service_fee'].apply(lambda x: remove_dollar_sign(x))

Index(['id', 'name', 'host_id', 'host_identity_verified', 'host_name',
       'neighbourhood_group', 'neighbourhood', 'lat', 'long', 'country',
       'country_code', 'instant_bookable', 'cancellation_policy', 'room_type',
       'construction_year', 'price', 'service_fee', 'minimum_nights',
       'number_of_reviews', 'last_review', 'reviews_per_month',
       'review_rate_number', 'calculated_host_listings_count',
       'availability_365', 'house_rules', 'license'],
      dtype='str')


### Table Creation

In [3]:
conn = duckdb.connect('airbnb.db')
try:
    conn.execute("SELECT 1 FROM airbnb_data LIMIT 1")
except duckdb.Error:
    conn.execute("CREATE TABLE airbnb_data AS SELECT * FROM read_csv_auto('Airbnb_Open_Data2.csv')")

### Categorical Distribution of data

In [4]:
conn.sql("""
SELECT distinct neighbourhood_group, count(*) as count_neighbourhood_group
FROM airbnb_data
         group by 1
         order by 2 desc
""")

┌─────────────────────┬───────────────────────────┐
│ neighbourhood_group │ count_neighbourhood_group │
│       varchar       │           int64           │
├─────────────────────┼───────────────────────────┤
│ Manhattan           │                     43792 │
│ Brooklyn            │                     41842 │
│ Queens              │                     13267 │
│ Bronx               │                      2712 │
│ Staten Island       │                       955 │
│ NULL                │                        29 │
│ manhatan            │                         1 │
│ brookln             │                         1 │
└─────────────────────┴───────────────────────────┘

In [5]:
conn.sql("""
SELECT distinct instant_bookable, count(*)
FROM airbnb_data
         group by 1
         order by 2 desc
""")

┌──────────────────┬──────────────┐
│ instant_bookable │ count_star() │
│     boolean      │    int64     │
├──────────────────┼──────────────┤
│ false            │        51474 │
│ true             │        51020 │
│ NULL             │          105 │
└──────────────────┴──────────────┘

In [6]:
conn.sql("""
SELECT distinct cancellation_policy, count(*) as count_cancellation_policy
FROM airbnb_data
         group by 1
         order by 2 desc
""")

┌─────────────────────┬───────────────────────────┐
│ cancellation_policy │ count_cancellation_policy │
│       varchar       │           int64           │
├─────────────────────┼───────────────────────────┤
│ moderate            │                     34343 │
│ strict              │                     34106 │
│ flexible            │                     34074 │
│ NULL                │                        76 │
└─────────────────────┴───────────────────────────┘

In [7]:
conn.sql("""
SELECT distinct room_type, count(*) as count_room_type
FROM airbnb_data
         group by 1
         order by 2 desc
""")

┌─────────────────┬─────────────────┐
│    room_type    │ count_room_type │
│     varchar     │      int64      │
├─────────────────┼─────────────────┤
│ Entire home/apt │           53701 │
│ Private room    │           46556 │
│ Shared room     │            2226 │
│ Hotel room      │             116 │
└─────────────────┴─────────────────┘

In [8]:
conn.sql("""
SELECT distinct construction_year, count(*) as count
FROM airbnb_data
         group by 1
         order by 1 desc
""")

┌───────────────────┬───────┐
│ construction_year │ count │
│      double       │ int64 │
├───────────────────┼───────┤
│            2022.0 │  5134 │
│            2021.0 │  5039 │
│            2020.0 │  5158 │
│            2019.0 │  5201 │
│            2018.0 │  5057 │
│            2017.0 │  5066 │
│            2016.0 │  5017 │
│            2015.0 │  5094 │
│            2014.0 │  5243 │
│            2013.0 │  5018 │
│            2012.0 │  5131 │
│            2011.0 │  5058 │
│            2010.0 │  5155 │
│            2009.0 │  5166 │
│            2008.0 │  5225 │
│            2007.0 │  5106 │
│            2006.0 │  5223 │
│            2005.0 │  5132 │
│            2004.0 │  5037 │
│            2003.0 │  5125 │
│              NULL │   214 │
└───────────────────┴───────┘
  21 rows         2 columns

### Numerical Distribution of data

In [9]:
df[['minimum_nights', 'number_of_reviews', 'reviews_per_month', 'review_rate_number', 'calculated_host_listings_count', 'availability_365']].describe()

,minimum_nights,number_of_reviews,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365
count,102190.000000,102416.000000,86720.000000,102273.000000,102280.000000,102151.000000
mean,8.174518,27.483743,1.374022,3.279106,7.936605,141.179284
std,30.543457,49.508954,1.746621,1.284657,32.218780,135.387040
min,1.000000,0.000000,0.010000,1.000000,1.000000,0.000000
25%,2.000000,1.000000,0.220000,2.000000,1.000000,4.000000
50%,3.000000,7.000000,0.740000,3.000000,1.000000,96.000000
75%,5.000000,30.000000,2.000000,4.000000,2.000000,269.000000
max,5645.000000,1024.000000,90.000000,5.000000,332.000000,3677.000000


In [10]:
df.to_csv('Airbnb_Open_Data2.csv')

# Data Visualization

I am going to create a new table, specifically will be used for visualization. For visualization, there are python libraries like matplotlib, seaborn, or ggplot2. But I'm going to use Streamlit